In [2]:
import numpy as np
from tensorflow.keras.datasets import mnist

# ==========================================
# 1. HÀM ĐỌC TRỌNG SỐ TỪ FILE TXT VỪA TRÍCH XUẤT
# ==========================================
def load_weights(filename, shape):
    # Đọc mảng 1D từ file và định hình lại (reshape) theo đúng shape của Tensor
    return np.loadtxt(filename, delimiter=',').reshape(shape)

# Load toàn bộ trọng số (Shape này lấy chuẩn từ báo cáo của bạn)
w1 = load_weights("conv1_weights.txt", (3, 3, 1, 8))
b1 = load_weights("conv1_bias.txt", (8,))
gamma1 = load_weights("bn1_gamma.txt", (8,))
beta1 = load_weights("bn1_beta.txt", (8,))
mean1 = load_weights("bn1_mean.txt", (8,))
var1 = load_weights("bn1_var.txt", (8,))

w2 = load_weights("conv2_weights.txt", (3, 3, 8, 16))
b2 = load_weights("conv2_bias.txt", (16,))
gamma2 = load_weights("bn2_gamma.txt", (16,))
beta2 = load_weights("bn2_beta.txt", (16,))
mean2 = load_weights("bn2_mean.txt", (16,))
var2 = load_weights("bn2_var.txt", (16,))

w_dense = load_weights("dense_weights.txt", (400, 10))
b_dense = load_weights("dense_bias.txt", (10,))

# ==========================================
# 2. XÂY DỰNG CÁC TOÁN TỬ BẰNG VÒNG LẶP FOR (Hardware-friendly)
# ==========================================
# 2.1. Phép tích chập (Conv2D)
def conv2d(input_matrix, kernel, bias):
    h_in, w_in, c_in = input_matrix.shape
    k_h, k_w, _, c_out = kernel.shape
    h_out, w_out = h_in - k_h + 1, w_in - k_w + 1
    output = np.zeros((h_out, w_out, c_out))

    # 3 vòng lặp for mô phỏng chính xác cách phần cứng sẽ nhặt dữ liệu
    for c in range(c_out):
        for i in range(h_out):
            for j in range(w_out):
                region = input_matrix[i:i+k_h, j:j+k_w, :]
                output[i, j, c] = np.sum(region * kernel[:, :, :, c]) + bias[c]
    return output

# 2.2. Chuẩn hóa theo lô (Batch Normalization)
def batch_norm(x, gamma, beta, mean, var, eps=1e-3):
    return gamma * (x - mean) / np.sqrt(var + eps) + beta

# 2.3. Hàm kích hoạt ReLU
def relu(x):
    return np.maximum(0, x)

# 2.4. Lấy mẫu cực đại (MaxPooling2D)
def max_pool2d(x, pool_size=2):
    h_in, w_in, c = x.shape
    h_out, w_out = h_in // pool_size, w_in // pool_size
    output = np.zeros((h_out, w_out, c))
    for c_i in range(c):
        for i in range(h_out):
            for j in range(w_out):
                region = x[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size, c_i]
                output[i, j, c_i] = np.max(region)
    return output

# ==========================================
# 3. THỰC THI INFERENCE (SUY LUẬN)
# ==========================================
print("Đang lấy bức ảnh đầu tiên từ tập test...")
(_, _), (X_test, y_test) = mnist.load_data()

# Ép kiểu ảnh xám và chuẩn hóa về [0, 1] y hệt bước tiền xử lý
# Yêu cầu người dùng nhập một số từ bàn phím
while True:
    try:
        index = int(input("Nhập chỉ số bức ảnh bạn muốn test (từ 0 đến 9999): "))
        if 0 <= index <= 9999:
            break
        else:
            print("Vui lòng chỉ nhập số trong khoảng từ 0 đến 9999!")
    except ValueError:
        print("Vui lòng nhập một số nguyên hợp lệ!")

# Ép kiểu ảnh xám và chuẩn hóa lấy theo vị trí 'index' vừa nhập
img = X_test[index].reshape(28, 28, 1).astype('float32') / 255.0
true_label = y_test[index]

print(f"\nĐã chọn bức ảnh thứ {index}.")
print(f"Nhãn thực tế của ảnh: {true_label}")
print("Đang chạy lan truyền thuận (Forward Pass) thuần Toán học...\n")
# Chạy qua Lớp 1
x = conv2d(img, w1, b1)
x = batch_norm(x, gamma1, beta1, mean1, var1)
x = relu(x)
x = max_pool2d(x)

# Chạy qua Lớp 2
x = conv2d(x, w2, b2)
x = batch_norm(x, gamma2, beta2, mean2, var2)
x = relu(x)
x = max_pool2d(x)

# Lớp Flatten (Duỗi mảng 3D thành 1D)
x_flat = x.flatten()

# Lớp Dense (Tính Logits tuyến tính, không dùng Softmax)
logits = np.dot(x_flat, w_dense) + b_dense

print("Giá trị Logits đầu ra:")
print(logits)

# Dùng hàm Argmax thân thiện với phần cứng để chốt kết quả
predicted_label = np.argmax(logits)
print(f"\n=> KẾT QUẢ DỰ ĐOÁN CỦA MÔ HÌNH: {predicted_label}")
if predicted_label == true_label:
    print("✅ CHÍNH XÁC! Suy luận NumPy hoạt động hoàn hảo.")
else:
    print("❌ SAI LỆCH! Cần kiểm tra lại trọng số.")

Đang lấy bức ảnh đầu tiên từ tập test...

Đã chọn bức ảnh thứ 30.
Nhãn thực tế của ảnh: 3
Đang chạy lan truyền thuận (Forward Pass) thuần Toán học...

Giá trị Logits đầu ra:
[ -9.33331399 -10.15938808   3.81754778  14.76346546  -4.12547398
  -0.74932523 -11.51662532   1.62976632  -1.07231936   0.85440294]

=> KẾT QUẢ DỰ ĐOÁN CỦA MÔ HÌNH: 3
✅ CHÍNH XÁC! Suy luận NumPy hoạt động hoàn hảo.


In [3]:
flat_img = img.flatten()
np.savetxt("input_image.txt", flat_img, fmt='%f', delimiter=',')
print("Đã lưu ảnh để C/C++ đọc!")

Đã lưu ảnh để C/C++ đọc!
